# 02 — Calendars, the registry and algebra

477 calendars ship inside the wheel. This notebook shows how to find them, what they know
about themselves, and above all how to **combine** them — the part where the vocabulary
traps everybody.

In [1]:
import numpy as np
import pandas as pd

import better_calendar as bcal
from better_calendar import Calendar

print(f"{len(bcal.list())} resolvable calendars")

479 resolvable calendars


## 1. Finding a calendar

Identifiers are **namespaced**. A bare four-letter name is an ISO-10383 MIC; everything
else carries a prefix, so `XNYS` cannot be confused with a country.

In [2]:
from collections import Counter

manifest = bcal.calendars.snapshot.load_manifest()
families = Counter(
    name.split(":")[0] if ":" in name else "MIC (exchange)" for name in bcal.list()
)
pd.DataFrame(sorted(families.items(), key=lambda kv: -kv[1]), columns=["prefix", "count"])

,prefix,count
0,country,251
1,wk,76
2,ql,75
3,MIC (exchange),57
4,fin,12
5,rate,4
6,exchange,3
7,crypto,1


In [3]:
# A sample of each family.
for prefix, examples in [
    ("MIC", ["XNYS", "XLON", "XPAR", "XTKS", "XTAE"]),
    ("country:", ["country:FR", "country:US", "country:US-NY", "country:JP"]),
    ("fin:", ["fin:TARGET2", "fin:NYB", "fin:LNB"]),
    ("rate:", ["rate:SOFR", "rate:ESTR", "rate:SONIA"]),
    ("others", ["crypto:24x7", "weekday", "ql:UnitedStates.NYSE", "wk:FR"]),
]:
    print(f"{prefix:10s} {', '.join(examples)}")

MIC        XNYS, XLON, XPAR, XTKS, XTAE
country:   country:FR, country:US, country:US-NY, country:JP
fin:       fin:TARGET2, fin:NYB, fin:LNB
rate:      rate:SOFR, rate:ESTR, rate:SONIA
others     crypto:24x7, weekday, ql:UnitedStates.NYSE, wk:FR


Aliases cover the names people actually say out loud. They live in a single declarative
table, never hardcoded.

In [4]:
rows = []
for alias in ("NYSE", "NASDAQ", "TARGET", "EUR", "GBP", "USD", "SONIA", "ESTR", "FR", "CRYPTO"):
    calendar = bcal.get(alias)
    rows.append({"alias": alias, "resolves to": calendar.name, "holidays": len(calendar.holidays)})
pd.DataFrame(rows).set_index("alias")

,resolves to,holidays
alias,,
NYSE,XNYS,1227
NASDAQ,XNYS,1227
TARGET,fin:TARGET2,536
EUR,fin:TARGET2,536
GBP,fin:LNB,1055
USD,fin:NYB,1376
SONIA,fin:LNB,1055
ESTR,fin:TARGET2,536
FR,country:FR,1120


In [5]:
# An alias and its target are the *same object* — memoisation survives resolution.
bcal.get("NYSE") is bcal.get("XNYS")

True

An unknown name does not return `None`: it raises, with suggestions.

In [6]:
try:
    bcal.get("XNYZ")
except bcal.UnknownCalendarError as exc:
    print(exc)

Unknown calendar 'XNYZ'. Did you mean: 'XNZE', 'XNYS'? Use better_calendar.list() to see everything available.


## 2. Provenance

Every calendar knows where it came from: which source, which **version** of that source,
over what horizon, and a digest of the contents. That is what makes a settlement date
auditable six months later.

In [7]:
bcal.describe("rate:SOFR")

{'name': 'rate:SOFR',
 'weekmask': 'Mon Tue Wed Thu Fri',
 'bounds': ['1970-01-01', '2100-12-31'],
 'tz': None,
 'session_start': '00:00:00',
 'holidays': 1473,
 'business_days': 32704,
 'provider': 'quantlib',
 'provider_version': '1.43',
 'hash': 'e1ce5112f1eb5ae7302b35ed98698941bc0f40a9',
 'requested': 'rate:SOFR',
 'canonical': 'rate:SOFR'}

The important part: `provider_version` is frozen. Upgrading `QuantLib` on the machine
changes **nothing** about the answer — the data comes from a committed file, not from a
call into the upstream library.

In [8]:
import sys

# No provider is imported, even after querying 400 calendars.
for name in list(bcal.list())[:400]:
    bcal.get(name)
[m for m in ("exchange_calendars", "holidays", "QuantLib", "workalendar") if m in sys.modules]

[]

## 3. What a calendar holds

In [9]:
nyse = bcal.get("XNYS")
print("name          :", nyse.name)
print("weekmask      :", nyse.weekmask)
print("timezone      :", nyse.tz)
print("bounds        :", nyse.bounds)
print("holidays      :", len(nyse.holidays))
print("business days :", len(nyse.good_days()))
print("\nfirst 5 holidays of 2026:")
print(list(nyse.holidays_between("2026-01-01", "2027-01-01")[:5].strftime("%Y-%m-%d (%a)")))

name          : XNYS
weekmask      : Mon Tue Wed Thu Fri
timezone      : America/New_York
bounds        : (datetime.date(1970, 1, 1), datetime.date(2100, 12, 31))
holidays      : 1227
business days : 32950

first 5 holidays of 2026:
['2026-01-01 (Thu)', '2026-01-19 (Mon)', '2026-02-16 (Mon)', '2026-04-03 (Fri)', '2026-05-25 (Mon)']


Not every calendar runs Monday to Friday. Tel Aviv trades Sunday to Thursday, and the
weekmask is **derived from the actual sessions**, not assumed:

In [10]:
tase = bcal.get("XTAE")
print("XTAE weekmask :", tase.weekmask)
print("Sunday 2 August 2026 a business day?", tase.is_bday("2026-08-02"))
print("Friday 31 July 2026 a business day? ", tase.is_bday("2026-07-31"))

XTAE weekmask : Mon Tue Wed Thu Sun
Sunday 2 August 2026 a business day? True
Friday 31 July 2026 a business day?  False


## 4. The algebra, and the vocabulary trap

**This is the most treacherous corner of the library.** "The union of two calendars" means
the opposite thing depending on whether you are thinking in **business days** or in
**holidays**.

The library settles it: everything is named after business days.

| Expression | Meaning |
|---|---|
| `a & b` | good in **both** — so the **union of the holidays** ← the settlement case |
| `a \| b` | good in **at least one** |
| `a - b` | good in `a`, not in `b` |
| `a ^ b` | good in **exactly one** |

Take two dates where New York and the euro area disagree: 3 July 2026 (Independence Day
observed, NY closed) and 6 April 2026 (Easter Monday, TARGET2 closed).

In [11]:
ny = bcal.get("XNYS")
eur = bcal.get("fin:TARGET2")

dates = ["2026-07-03", "2026-04-06", "2026-07-02"]
table = {
    "XNYS": {d: ny.is_bday(d) for d in dates},
    "fin:TARGET2": {d: eur.is_bday(d) for d in dates},
    "a & b  (both)": {d: (ny & eur).is_bday(d) for d in dates},
    "a | b  (at least one)": {d: (ny | eur).is_bday(d) for d in dates},
    "a - b  (NY only)": {d: (ny - eur).is_bday(d) for d in dates},
    "a ^ b  (exactly one)": {d: (ny ^ eur).is_bday(d) for d in dates},
}
pd.DataFrame(table).T

,2026-07-03,2026-04-06,2026-07-02
XNYS,False,True,True
fin:TARGET2,True,False,True
a & b (both),False,False,True
a | b (at least one),True,True,True
a - b (NY only),False,True,False
a ^ b (exactly one),True,True,False


The concrete case: a cash flow between New York and the euro area can only move on a day
**both** centres are open. That is `&`, and it really is the union of the holiday sets.

Somebody describing that as "the union of the calendars" is thinking in holidays and would
reach for `|`. Hence the verbose aliases, which leave no doubt in review:

In [12]:
settlement = Calendar.all_open([ny, eur])
print("derived name :", settlement.name)
print("2026-07-02 + 1 settlement day ->", settlement.offset("2026-07-02", 1))
print("same as & ?", settlement == (ny & eur))

derived name : (XNYS & fin:TARGET2)
2026-07-02 + 1 settlement day -> 2026-07-06
same as & ? True


A composite is an ordinary `Calendar`: frozen, hashable, reusable in offsets and schedules.
Its bounds are the intersection of the operands'.

In [13]:
composite = ny & eur & bcal.get("XLON")
print("name        :", composite.name)
print("bounds      :", composite.bounds)
print("good days   :", len(composite.good_days()))
print("hashable    :", hash(composite) == hash(ny & eur & bcal.get("XLON")))

name        : ((XNYS & fin:TARGET2) & XLON)
bounds      : (datetime.date(1970, 1, 1), datetime.date(2100, 12, 31))
good days   : 32313
hashable    : True


### Heterogeneous weekends

The implementation is set algebra on good days, never a merge of weekmask strings. That is
what makes crossing a Sunday–Thursday calendar with a Monday–Friday one work for free:

In [14]:
gulf = Calendar("gulf", weekmask="Sun Mon Tue Wed Thu")
mon_fri = bcal.get("weekday")

print("intersection :", (mon_fri & gulf).weekmask)   # the overlap
print("union        :", (mon_fri | gulf).weekmask)   # everything but Saturday
print()
for day in ("2026-07-31", "2026-08-01", "2026-08-02"):
    what = pd.Timestamp(day).strftime("%a")
    print(f"{day} ({what})  &={(mon_fri & gulf).is_bday(day)!s:5s}  |={(mon_fri | gulf).is_bday(day)}")

intersection : Mon Tue Wed Thu
union        : Mon Tue Wed Thu Fri Sun



2026-07-31 (Fri)  &=False  |=True
2026-08-01 (Sat)  &=False  |=False


2026-08-02 (Sun)  &=False  |=True


### A composite's timezone

A composite spanning two zones has **no** instant semantics, so it loses its timezone — and
saying so early beats a wrong answer later.

In [15]:
print("XNYS tz                :", ny.tz)
print("XPAR tz                :", bcal.get("XPAR").tz)
print("XNYS & XPAR            :", (ny & bcal.get("XPAR")).tz, "  <- disagreement: the zone drops")

# Two calendars declaring the *same* zone keep it.
desk_ny = Calendar("desk:ny", tz="America/New_York", holidays=["2026-11-27"])
print("XNYS & desk:ny         :", (ny & desk_ny).tz)

# The comparison is on the IANA name, not the offset: Paris and Amsterdam share an offset
# but are two distinct identifiers, so the composite tells them apart.
print("XPAR & XAMS            :", (bcal.get("XPAR") & bcal.get("XAMS")).tz,
      f'  ({bcal.get("XPAR").tz} vs {bcal.get("XAMS").tz})')

XNYS tz                : America/New_York
XPAR tz                : Europe/Paris
XNYS & XPAR            : None   <- disagreement: the zone drops
XNYS & desk:ny         : America/New_York


XPAR & XAMS            : None   (Europe/Paris vs Europe/Amsterdam)


## 5. Deriving a calendar without forking

`with_holidays` / `without_holidays` return a **new** calendar. The original is frozen and
never moves.

In [16]:
desk = ny.with_holidays(["2026-11-27"], name="desk:us")   # day after Thanksgiving
print("desk           :", desk.is_bday("2026-11-27"))
print("XNYS untouched :", ny.is_bday("2026-11-27"))

desk           : False
XNYS untouched : True


In [17]:
# And it can be registered under a name, so every call site finds it.
bcal.register("desk:us", desk)
print(bcal.get("desk:us").is_bday("2026-11-27"))
print("listed :", "desk:us" in bcal.list())
bcal.unregister("desk:us")

False
listed : True


## Recap

| Call | Role |
|---|---|
| `bcal.get(name)` | resolve an identifier or alias, memoised |
| `bcal.list(provider=)` | everything resolvable |
| `bcal.describe(name)` | provenance: source, version, bounds, digest |
| `bcal.register` / `unregister` | install your own calendar |
| `a & b`, `a \| b`, `a - b`, `a ^ b` | algebra, named after **business** days |
| `Calendar.all_open` / `any_open` | the same, spelled unambiguously |
| `with_holidays` / `without_holidays` | derive without mutating |

**Next:** [03 — Offsets, tenors and settlement](03-offsets-tenors-settlement.ipynb)